# 01. Exploratory Data Analysis & Air Quality Forecasting Benchmark
## Python Capstone Project | Lab 6 Track C
**Domain:** Environmental Atmospheric Science & Machine Learning  
**Target:** Continuous Daily Air Quality Index (AQI) for Delhi NCR, India  
**Formulation:** Strictly Prior-Only Next-Day Forecasting: $\mathbf{X}(t-14, \dots, t-1) \longrightarrow 	ext{AQI}(t)$  

---

### A. Problem Statement
Ambient air pollution in the Delhi National Capital Region is among the most severe in the world, regularly exceeding national standards by up to an order of magnitude during winter stagnation events. Accurate 24-hour ahead forecasting of the Air Quality Index (AQI) is essential for public health advisories, school closures, and regulatory interventions.

However, standard time-series forecasting benchmarks often suffer from subtle data leakage (such as using unlagged contemporaneous pollutant measurements to predict the same day's AQI). The objective of this capstone is to establish a **strictly leak-free, reproducible comparative evaluation** between:
1. Classical Climatological calendar baselines
2. Standalone tree ensembles (XGBoost, Extra Trees)
3. Deep sequence neural networks in PyTorch (Bi-LSTM, Bi-GRU, Dilated TCN)
4. Heterogeneous stacked ensembles (SE-1 and SE-2) with Out-Of-Fold meta-learning


---
### B. Dataset Overview & Provenance
Two primary open-access environmental datasets are integrated:
1. **Criteria Air Pollutants:** Central Pollution Control Board (CPCB) India National Ambient Air Quality Network (), spanning 26 Indian cities from 2015 to 2020.
2. **Atmospheric Meteorology:** European Centre for Medium-Range Weather Forecasts (ECMWF) ERA5 atmospheric reanalysis via Open-Meteo Historical Archive API at Delhi coordinates (28.6139°N, 77.2090°E).


In [ ]:
import sys
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure visual style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.size"] = 10

# Load raw datasets
df_pollutants = pd.read_csv("../data/raw/city_day.csv")
df_weather = pd.read_csv("../data/raw/weather_daily.csv")

print(f"Raw CPCB dataset shape: {df_pollutants.shape}")
print(f"Cities in CPCB network: {df_pollutants['City'].nunique()} cities")
print(f"Raw ERA5 weather dataset shape: {df_weather.shape}")


---
### C & D. Data Dimensions, Date Range & Delhi Filtering
We isolate Delhi, which represents the most comprehensive and continuous monitoring record in the CPCB network (2,009 consecutive daily records from January 1, 2015 to July 1, 2020).


In [ ]:
# Filter for Delhi and parse dates
df_delhi = df_pollutants[df_pollutants["City"] == "Delhi"].copy()
df_delhi["Date"] = pd.to_datetime(df_delhi["Date"])
df_delhi = df_delhi.sort_values("Date").reset_index(drop=True)

df_weather["Date"] = pd.to_datetime(df_weather["Date"])
df_merged = pd.merge(df_delhi, df_weather, on="Date", how="inner").sort_values("Date").reset_index(drop=True)

print(f"Delhi merged daily observations: {len(df_merged)} rows")
print(f"Date Range: {df_merged['Date'].min().strftime('%Y-%m-%d')} to {df_merged['Date'].max().strftime('%Y-%m-%d')}")
print(f"Columns ({len(df_merged.columns)}): {df_merged.columns.tolist()}")


---
### E. Missing-Value Analysis
Air quality sensor feeds experience intermittent dropouts due to power fluctuations and maintenance. We inspect the missing value profile for Delhi to ensure appropriate, non-leaking imputation.


In [ ]:
cols_of_interest = ["PM2.5", "PM10", "NO2", "SO2", "CO", "O3", "Temperature", "Humidity", "Wind_Speed", "AQI"]
missing_summary = pd.DataFrame({
    "Missing_Count": df_merged[cols_of_interest].isnull().sum(),
    "Missing_Pct": (df_merged[cols_of_interest].isnull().sum() / len(df_merged)) * 100
})
print("Delhi Missing-Value Summary:")
print(missing_summary)


---
### F. Criteria Pollutant Distributions
We visualize the empirical distribution of the criteria pollutants ({2.5}, PM_{10}, NO_2, SO_2, CO, O_3$). Particulate matter ({2.5}$ and {10}$) exhibits extreme right skewness due to episodic seasonal spikes.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
criteria = ["PM2.5", "PM10", "NO2", "SO2", "CO", "O3"]
colors = ["#ef4444", "#f97316", "#eab308", "#22c55e", "#06b6d4", "#8b5cf6"]

for ax, pol, col in zip(axes.flat, criteria, colors):
    sns.histplot(df_merged[pol].dropna(), kde=True, ax=ax, color=col, bins=30)
    missing = df_merged[pol].isnull().sum()
    ax.set_title(f"{pol} (Missing: {missing})", fontweight="bold")
    ax.set_xlabel("Concentration (µg/m³ or mg/m³)")

plt.suptitle("Figure 02: Criteria Air Pollutant Distributions in Delhi (2015-2020)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../results/figures/02_missing_values_and_distributions.png", dpi=300)
plt.show()


---
### G. AQI Time-Series Visualization & Indian NAQI Categories
The Indian Central Pollution Control Board (CPCB) defines six AQI categories:
* **Good:** 0–50
* **Satisfactory:** 51–100
* **Moderate:** 101–200
* **Poor:** 201–300
* **Very Poor:** 301–400
* **Severe:** > 400

Below is the complete 5.5-year trajectory with the chronological 80:20 train/test split indicated.


In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df_merged["Date"], df_merged["AQI"], color="#1f2937", linewidth=1.2, label="Daily CPCB AQI")
plt.axhline(50, color="#28a745", linestyle="--", alpha=0.7, label="Good (0-50)")
plt.axhline(100, color="#8bc34a", linestyle="--", alpha=0.7, label="Satisfactory (51-100)")
plt.axhline(200, color="#ffc107", linestyle="--", alpha=0.7, label="Moderate (101-200)")
plt.axhline(300, color="#fd7e14", linestyle="--", alpha=0.7, label="Poor (201-300)")
plt.axhline(400, color="#dc3545", linestyle="--", alpha=0.7, label="Very Poor (301-400)")
plt.axvline(pd.to_datetime("2019-05-27"), color="#4f46e5", linestyle="-", linewidth=2, label="Train / Held-Out Test Split (80:20)")
plt.title("Figure 01: Delhi Daily AQI Timeline (2015-2020) with CPCB Severity Thresholds", fontsize=13, fontweight="bold")
plt.xlabel("Date")
plt.ylabel("Air Quality Index (AQI)")
plt.legend(loc="upper right", ncol=2, frameon=True, fontsize=9)
plt.tight_layout()
plt.savefig("../results/figures/01_aqi_timeline.png", dpi=300)
plt.show()


---
### H. Correlation Heatmap: Pollutants, Meteorology & AQI
We examine linear relationships across all variables. Notice that {2.5}$ and {10}$ show the strongest positive correlation with AQI ( > 0.85$), while temperature and wind speed show negative correlations due to atmospheric dispersion.


In [ ]:
plt.figure(figsize=(10, 8))
corr_cols = ["AQI", "PM2.5", "PM10", "NO2", "SO2", "CO", "O3", "Temperature", "Humidity", "Wind_Speed"]
corr = df_merged[corr_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", cbar=True, vmin=-1, vmax=1, linewidths=0.5)
plt.title("Figure 03: Correlation Matrix: Criteria Pollutants, Meteorology & AQI", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("../results/figures/03_correlation_heatmap.png", dpi=300)
plt.show()


---
### I. Seasonality & Temporal Patterns
Delhi exhibits strong seasonal cyclicity:
* **Winter (Nov–Feb):** Severe pollution driven by shallow boundary layers, calm winds, and temperature inversions.
* **Monsoon (Jul–Sep):** Cleanest air due to heavy precipitation wash-out and elevated wind speeds.


In [ ]:
df_merged["Month"] = df_merged["Date"].dt.month_name()
month_order = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]

plt.figure(figsize=(12, 4))
sns.boxplot(data=df_merged, x="Month", y="AQI", order=month_order, palette="viridis")
plt.title("Delhi Monthly AQI Seasonality (2015-2020)", fontweight="bold")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


---
### J. Explanation of the Forecasting Target
The modeling target is the continuous next-day scalar value $	ext{AQI}(t)$.
Crucially, **no same-day measurements from day * are permitted in the input feature space:
8735\mathbf{X}_{	ext{prior}} = \{x_{t-1}, x_{t-2}, \dots, x_{t-14}\} \longrightarrow 	ext{AQI}(t)8735

### K. Feature Engineering: Lags, Rolling Statistics & EWMA
1. **Calendar & Cyclic:** Day of week, month, day of year, weekend flag, Indian meteorological season, and cyclical sine/cosine transforms ($\sin/\cos$ of day of year, month, day of week).
2. **Historical Lags:** For AQI, 6 criteria pollutants, and 3 weather variables, lags at **-1, t-2, t-3, t-7* are computed.
3. **Shifted Rolling Windows:** 7-day and 14-day rolling mean, std, min, max, and 7-day EWMA are computed on series **strictly shifted by 1 day**:
   8735	ext{roll\_mean}_7(t) = rac{1}{7}\sum_{k=1}^7 x_{t-k}8735
   Total engineered tabular features: **114**.

### L. 3D Sequence Tensor Construction (=14$)
For deep sequence models (Bi-LSTM, Bi-GRU, TCN), a sliding temporal window of length =14$ days across 10 continuous channels is constructed:
8735\mathbf{S}(t) \in \mathbb{R}^{14 	imes 10}, \quad 	ext{covering indices } [t-14, \dots, t-1]8735

### M. Train/Test Chronological Split (80:20)
* **Training Partition:** 1,593 sequence samples (2015-01-15 to 2019-05-26).
* **Held-Out Test Partition:** 402 sequence samples (2019-05-27 to 2020-07-01).
* **Scaler Isolation:**  instances for tabular features, sequence features, and target are fitted **strictly on the training partition**. Test samples are transformed using frozen training scalers.


In [ ]:
with open("../data/processed/dataset_bundle.pkl", "rb") as f:
    bundle = pickle.load(f)

print("Processed Dataset Bundle Summary:")
print(f"  X_tab_train shape: {bundle['X_tab_train'].shape}  |  X_seq_train shape: {bundle['X_seq_train'].shape}")
print(f"  X_tab_test  shape: {bundle['X_tab_test'].shape}   |  X_seq_test  shape: {bundle['X_seq_test'].shape}")
print(f"  y_train shape:     {bundle['y_train'].shape}       |  y_test shape:     {bundle['y_test'].shape}")
print(f"  Total tabular features: {len(bundle['tabular_cols'])}")
print(f"  Sequence channels:      {bundle['seq_cols']}")


---
### N. Model Lineup
1. **Historical Climatology:** Non-parametric calendar baseline ($\pm 7569Xday historical moving window).
2. **XGBoost (Standalone):** Gradient Boosted Decision Trees (200 estimators, depth 5, lr 0.04) on 114 tabular features.
3. **Extra Trees (Standalone):** 100 extremely randomized trees (depth 8) on tabular features.
4. **Bi-LSTM (PyTorch):** Bidirectional LSTM (1 layer, 64 hidden units) on 14-day sequence tensor.
5. **Bi-GRU (PyTorch):** Bidirectional GRU (1 layer, 64 hidden units) on 14-day sequence tensor.
6. **Dilated TCN (PyTorch):** Temporal Convolutional Network with causal dilated 1D convs (channels [32, 32, 64, 64], dilations 1, 2, 4, 8) and residual blocks.
7. **SE-1 (Stacked Ensemble):** XGBoost + Bi-LSTM + TCN $	o$ Extra Trees Meta-Learner.
8. **SE-2 (Stacked Ensemble):** XGBoost + Bi-GRU + TCN $	o$ Extra Trees Meta-Learner.

### O. Evaluation Methodology
* **Out-Of-Fold Cross-Validation:** The stacking meta-learner is trained on 5-fold  OOF predictions to prevent meta-learner overfitting.
* **Metrics:** Root Mean Squared Error (RMSE), Mean Absolute Error (MAE), Mean Absolute Percentage Error (MAPE), Symmetric MAPE (sMAPE), Coefficient of Determination (^2$), and Inference Latency (ms/sample).
* **Multi-Seed Stability:** Evaluated across random seeds [42, 101, 2024] to test robustness.


In [ ]:
df_metrics = pd.read_csv("../results/metrics/model_comparison.csv")
df_stability = pd.read_csv("../results/metrics/repeated_runs_stability.csv")

print("Authoritative Benchmark Model Comparison (Held-Out Test N=402):")
print(df_metrics.to_string(index=False))

print("
Multi-Seed Stability (3 Seeds: Mean ± SD):")
print(df_stability.to_string(index=False))


In [ ]:
plt.figure(figsize=(12, 6))
df_plot = df_metrics.sort_values("RMSE", ascending=True).reset_index(drop=True)
colors_bar = ["#15803d" if m == "XGBoost" else "#3b82f6" if m.startswith("SE") else "#94a3b8" for m in df_plot["Model"]]
bars = plt.bar(df_plot["Model"], df_plot["RMSE"], color=colors_bar, width=0.55, edgecolor="#334155")

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1.2, f"{yval:.2f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.ylabel("Test RMSE (AQI Points) - Lower is Better", fontsize=12)
plt.title("Figure 04: Benchmark Test RMSE Comparison across Architectures (Held-out Test N=402)", fontsize=13, fontweight="bold")
plt.xticks(rotation=25, ha="right", fontsize=11)
plt.ylim(0, 115)
plt.axhline(34.58, color="#15803d", linestyle=":", label="XGBoost Standalone (Best: 34.58)")
plt.legend(frameon=True)
plt.tight_layout()
plt.savefig("../results/figures/04_model_rmse_comparison.png", dpi=300)
plt.show()


---
### Q. Actual vs. Predicted Visualizations
We overlay the ground truth actual AQI against the top point model (XGBoost) and the stacked ensembles (SE-1 and SE-2) over the full 402 held-out test days.


In [ ]:
df_preds = pd.read_csv("../results/predictions/test_predictions.csv")
d_test = pd.to_datetime(df_preds["Date"])

plt.figure(figsize=(14, 6))
plt.plot(d_test, df_preds["Actual_AQI"], color="#0f172a", linewidth=2.0, label="Actual Ground Truth AQI", alpha=0.9)
plt.plot(d_test, df_preds["Pred_XGBoost"], color="#16a34a", linestyle="--", linewidth=1.6, label="XGBoost Standalone (Top Point Model)")
plt.plot(d_test, df_preds["Pred_SE-2"], color="#2563eb", linewidth=1.6, label="SE-2 Ensemble (XGB+BiGRU+TCN)")
plt.plot(d_test, df_preds["Pred_SE-1"], color="#f59e0b", linewidth=1.5, label="SE-1 Ensemble (XGB+BiLSTM+TCN)")
plt.title("Figure 05: Test Period Forecast Trajectories vs Actual AQI (May 2019 - July 2020)", fontsize=13, fontweight="bold")
plt.xlabel("Date", fontsize=11)
plt.ylabel("Air Quality Index (AQI)", fontsize=11)
plt.legend(loc="upper right", frameon=True)
plt.tight_layout()
plt.savefig("../results/figures/05_actual_vs_predicted.png", dpi=300)
plt.show()


---
### R. Residual Error & Ensemble Analysis
We examine the residual distributions ( = y - \hat{y}$) and actual vs. predicted scatter agreement.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for m_name, col in [("XGBoost", "#16a34a"), ("SE-2", "#2563eb"), ("SE-1", "#f59e0b"), ("Bi-LSTM", "#dc2626")]:
    res = df_preds["Actual_AQI"] - df_preds[f"Pred_{m_name}"]
    sns.kdeplot(res, ax=ax1, label=f"{m_name} (SD={np.std(res):.1f})", color=col, linewidth=2.0)

ax1.axvline(0, color="black", linestyle="--", alpha=0.6)
ax1.set_title("Residual Error Distributions (y - y_hat)", fontweight="bold", fontsize=11)
ax1.set_xlabel("Error (AQI points)")
ax1.set_xlim(-120, 120)
ax1.legend()

ax2.scatter(df_preds["Actual_AQI"], df_preds["Pred_XGBoost"], color="#16a34a", alpha=0.4, label="XGBoost (R2=0.906)", s=20)
ax2.scatter(df_preds["Actual_AQI"], df_preds["Pred_SE-2"], color="#2563eb", alpha=0.4, label="SE-2 (R2=0.880)", s=20)
lims = [0, 500]
ax2.plot(lims, lims, color="black", linestyle="--", label="Perfect Agreement (y=x)", linewidth=1.5)
ax2.set_xlim(lims)
ax2.set_ylim(lims)
ax2.set_title("Actual vs Predicted Parity Plot", fontweight="bold", fontsize=11)
ax2.set_xlabel("Actual AQI")
ax2.set_ylabel("Predicted AQI")
ax2.legend()

plt.suptitle("Figure 06: Model Error & Residual Analysis on Held-Out Test Data", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../results/figures/06_ensemble_error_analysis.png", dpi=300)
plt.show()


---
### S. Final Recommendation & Capstone Conclusions
1. **Headline Finding:** Standalone **XGBoost** is the definitive top performing model for 24-hour ahead AQI point forecasting:
   * Lowest Test RMSE: **34.58**
   * Lowest Test MAE: **26.08**
   * Highest Test ^2$: **0.9063**
   * Ultra-low inference latency: **0.016 ms/sample** (16x faster than neural architectures)
   * Statistically superior to all other models (Diebold-Mariano  = 0.00054$).
2. **Ensemble Architecture Finding:** The stacked ensemble **SE-2 (XGBoost + Bi-GRU + TCN)** strongly outperforms SE-1 (RMSE 39.14 vs. 45.85, paired  = -8.74, p < 10^{-16}$) with 43% tighter multi-seed stability (SD 1.12 vs 1.96).
3. **Scientific Lesson:** Complex deep stacked ensembles do not automatically outperform gradient boosted decision trees when tabular lag and rolling features are engineered with domain precision. However, multi-model consensus provides higher variance stabilization and spike resilience under extreme regimes.
